In [1]:
# Run from the repo root so relative data/ paths resolve, wherever the notebook is launched.
import os
for _ in range(6):
    if os.path.exists('pyproject.toml'):
        break
    os.chdir('..')


# Data Preparation

Builds the month-end SET100 stock panel. For every stock and month it computes a forward-looking **realized-beta target** and **ten predictors** (three historical rolling-OLS betas plus seven firm/market characteristics). The heavy lifting lives in tested modules under `betaforecast/`, so the modelling notebook reuses the exact same code:

| Module | Role |
|---|---|
| `betaforecast.data.sources` | load & clean the raw inputs in `data/raw/` |
| `betaforecast.data.dividends` | cash-dividend yields from SETSMART → total returns |
| `betaforecast.data.returns` | daily / monthly total & excess returns |
| `betaforecast.data.predictors` | ME, DIVPAY, TO, MOM, IVOL, ILLIQ, DFY |
| `betaforecast.data.panel` | realized-beta target, rolling-OLS betas, complete-case cleaning |
| `betaforecast.data.build` | end-to-end build, descriptive tables, save panel |

**Note on prices:** the Refinitiv `close` is split/par-adjusted but **not** cash-dividend adjusted. `betaforecast.data.dividends` adds the cash-dividend yield (from SETSMART ex-dividend records) on each ex-date to form the daily **total** return, consistent with the total-return market proxy.

Run with `uv run jupyter lab` (installs the pinned environment).

In [2]:
import pandas as pd
import numpy as np

from betaforecast.data import sources as data_io, returns as R, panel as B, build as F

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## Data Sample
Raw inputs: daily total-return-adjusted prices, share volume, daily market cap, SET100 historical membership, the SET100 TRI market proxy, the ThaiBMA 1–3Y risk-free index, Thai Fama–French/q factors, the precomputed dividend-payout ratio, and the FRED default spread.

In [3]:
prices = data_io.load_prices()
membership = data_io.load_membership()
print('price obs:', f'{len(prices):,}', '| rics:', prices.ric.nunique(),
      '| dates:', prices.date.min().date(), '->', prices.date.max().date())
print('membership: ~', membership.groupby('date').ric.nunique().median(), 'constituents per day')

price obs: 1,120,139 | rics: 304 | dates: 2005-01-04 -> 2025-12-30
membership: ~ 100.0 constituents per day


## Return Calculation
Daily continuously compounded **total** returns ($r = r^{price} + D/P^{*}$ on ex-dates), daily excess returns ($r^e = r - r_f$), and monthly excess returns (sum of daily within the calendar month).

In [4]:
stock_daily, mkt_excess, rf, raw_mkt = R.build_daily_returns()
stock_daily.head()

[dividends] 5676 cash-dividend events applied to 285 stocks; capped 0 at 50%; dropped 0 (no usable price).


,ric,date,r_price,div_yield,r,rf,r_excess
0,A.BK,2005-01-05,0.0217,0.0000,0.0217,0.0009,0.0207
1,A.BK,2005-01-06,0.0282,0.0000,0.0282,-0.0004,0.0286
2,A.BK,2005-01-07,0.0069,0.0000,0.0069,-0.0005,0.0074
3,A.BK,2005-01-10,-0.0139,0.0000,-0.0139,0.0021,-0.0160
4,A.BK,2005-01-11,-0.0212,0.0000,-0.0212,-0.0029,-0.0183


## Panel Construction
Build the full stock-month panel: the 12-month-forward realized-beta target, the three rolling-OLS betas, and the seven firm/market characteristics.

In [5]:
res, *_ = B.build()
res[[B.TARGET] + B.PREDICTORS].describe().T

[dividends] 5676 cash-dividend events applied to 285 stocks; capped 0 at 50%; dropped 0 (no usable price).


,count,mean,std,min,25%,50%,75%,max
beta_realized_fwd,"50,767.0000",0.8248,0.4178,-4.3947,0.5346,0.7985,1.0864,3.6072
OLS_3M_D,"54,222.0000",0.8045,0.6075,-12.7193,0.4278,0.7540,1.1245,10.1527
OLS_1Y_D,"53,500.0000",0.8193,0.4266,-4.3947,0.5245,0.7938,1.0848,5.3143
OLS_5Y_M,"45,699.0000",1.0571,0.4770,-1.1970,0.7373,1.0278,1.3499,3.4695
ME,"55,716.0000",9.3798,1.7162,3.7414,8.2161,9.3684,10.4761,14.8113
DIVPAY,"66,484.0000",0.5642,4.8143,-18.8920,0.0000,0.3079,0.6289,263.4212
TO,"54,338.0000",5.9874,2.5118,-7.9687,4.3245,6.2584,7.8325,12.0018
MOM,"51,194.0000",-0.0043,0.4805,-3.4005,-0.2644,-0.0120,0.2415,4.4815
IVOL,"54,355.0000",-4.1964,0.5992,-9.1311,-4.5722,-4.2260,-3.8644,-0.5288
ILLIQ,"54,333.0000",0.0214,1.2658,0.0000,0.0000,0.0001,0.0006,271.0417


## Cleaning and Membership Filter
Keep active SET100 members over Apr 2008 – Dec 2024 that have all three historical betas, a valid target, **and** a complete set of the six firm predictors (complete-case; no imputation).

In [6]:
clean = B.cleaned_panel(res)
print(f'stock-months: {len(clean):,} | month-ends: {clean.index.get_level_values("ym").nunique()}'
      f' | avg stocks/month: {len(clean)/clean.index.get_level_values("ym").nunique():.1f}')
print('missing firm-predictor values (complete-case):', int(clean[B.FIRM].isna().sum().sum()))

stock-months: 17,887 | month-ends: 201 | avg stocks/month: 89.0
missing firm-predictor values (complete-case): 0


## Descriptive Statistics
Table: daily & monthly returns (active panel, active membership).

In [7]:
(F.returns_table() * 100).round(3)  # percent, except N

[dividends] 5676 cash-dividend events applied to 285 stocks; capped 0 at 50%; dropped 0 (no usable price).


N   Mean    Std      Min  Median     Max
A Stocks in SET100 408,000.0000 0.0300 1.2270 -13.4230  0.0850  8.4630
  SET100 Index     408,000.0000 0.0240 1.2540 -12.6850  0.0520  8.7880
  Risk-free rate   408,000.0000 0.0090 0.0460  -0.3370  0.0090  0.5640
B Stocks in SET100  20,100.0000 0.6080 6.7960 -42.5320  1.1540 22.2460
  SET100 Index      20,100.0000 0.4960 5.8650 -37.6910  0.8350 18.7240
  Risk-free rate    20,100.0000 0.1920 0.3650  -1.0930  0.1870  2.5680

In [8]:
stats = clean[[B.TARGET] + B.PREDICTORS].agg(['count', 'mean', 'std', 'min', 'median', 'max']).T
stats['count'] = stats['count'].astype(int)
stats

,count,mean,std,min,median,max
beta_realized_fwd,17887,0.9442,0.3773,-2.0505,0.9219,3.2181
OLS_3M_D,17887,0.9382,0.4888,-3.2649,0.9032,5.7707
OLS_1Y_D,17887,0.9432,0.3674,-1.4974,0.9230,3.1643
OLS_5Y_M,17887,1.0864,0.4118,-0.3757,1.0721,2.9025
ME,17887,10.7401,1.2931,5.6255,10.5941,14.4586
DIVPAY,17887,0.5534,1.8974,-10.3519,0.4345,68.1305
TO,17887,7.9338,1.5302,-0.8371,7.9903,12.0018
MOM,17887,0.0350,0.4254,-2.8600,0.0333,2.3782
IVOL,17887,-4.2779,0.4656,-8.8737,-4.2905,-1.7530
ILLIQ,17887,0.0003,0.0052,0.0000,0.0000,0.3873


## Estimation Sample (strict 12-month lookback)
Apply the strict sample-validity filter used by the model: keep a SET100 **member** month-end only if it is anchored by a complete, gap-free 12-month history of the ten predictors (interior lookback months may be non-member). These origin-months are the **estimation sample** used in all downstream modelling.

In [9]:
from betaforecast.features import sequences as S
tp = B.lstm_input_panel(res).reset_index().sort_values(['ric', 'ym'])
Xseq, yseq, seq_meta = S.build_sequences(tp)
origin = pd.DataFrame(Xseq[:, -1, :], columns=S.PREDICTORS)   # predictors at the forecast origin t
origin[S.TARGET] = yseq
est = origin[[S.TARGET] + S.PREDICTORS].agg(['count', 'mean', 'std', 'min', 'median', 'max']).T
est['count'] = est['count'].astype(int)
print(f'estimation sample: {len(seq_meta):,} member origin-months | '
      f'{seq_meta["ric"].nunique()} stocks | {seq_meta["ym"].nunique()} month-ends '
      f'({seq_meta["ym"].min()} to {seq_meta["ym"].max()})')
est

estimation sample: 16,285 member origin-months | 208 stocks | 190 month-ends (2009-03 to 2024-12)


,count,mean,std,min,median,max
beta_realized_fwd,16285,0.9443,0.3781,-2.0505,0.9177,3.2181
OLS_3M_D,16285,0.9450,0.4919,-3.2649,0.9092,5.7707
OLS_1Y_D,16285,0.9501,0.3677,-1.4974,0.9289,3.1643
OLS_5Y_M,16285,1.0898,0.4053,-0.2704,1.0748,2.9013
ME,16285,10.8245,1.2544,6.0624,10.6524,14.4586
DIVPAY,16285,0.5456,1.8875,-10.3519,0.4350,68.1305
TO,16285,8.0590,1.4308,0.5020,8.0852,12.0018
MOM,16285,0.0637,0.4010,-2.8600,0.0505,2.3782
IVOL,16285,-4.2976,0.4583,-8.8737,-4.3082,-1.7530
ILLIQ,16285,0.0001,0.0010,0.0000,0.0000,0.0975


## Save Processed Panel

In [10]:
out = clean[B.PREDICTORS + [B.TARGET, 'date', 'n_daily_month']].reset_index()
out['ym'] = out['ym'].astype(str)
out.to_parquet('data/processed/processed_data_panel.parquet')
print('saved data/processed/processed_data_panel.parquet', out.shape)

saved data/processed/processed_data_panel.parquet (17887, 15)
